# How to train an KAN using the Pipeline

## Import classes and define paths

In [ ]:
from pyLOM import NN
from pathlib import Path

import torch

import warnings
warnings.filterwarnings("ignore")

In [ ]:
DATA_DIR = Path.cwd().parent.parent.parent.parent / "Testsuite/DATA"
CASE_DIR = Path.cwd() / "results"

NN.create_results_folder(CASE_DIR / 'models')
NN.create_results_folder(CASE_DIR / 'hyperparameters')
NN.create_results_folder(CASE_DIR / 'plots')

## Define scalers if needed

Here, we create 2 minmax scalers, one for scaling the inputs, and other for the outputs. They will be passed to the dataset and the data will be automatically scaled

In [ ]:
input_scaler = NN.MinMaxScaler()
output_scaler = NN.MinMaxScaler()

## Create datasets
For this example, a dataset of airfoils generated with XFoil is used. As inputs, the model will receive the x and y coordinates of a point of the airfoil, the Reynolds number and the angle of attack; and the output will be the cp on that point

In [ ]:
dataset = NN.Dataset.load(
    DATA_DIR / 'AIRFOIL.h5',
    field_names=["cp"],
    add_mesh_coordinates=True,
    variables_names=["AoA", "Re"],
    inputs_scaler=input_scaler,
    outputs_scaler=output_scaler,
)
train_dataset, test_dataset = dataset.get_splits_by_parameters([0.8, 0.2])

After creating the datasets, we can see the shape of the tensors

In [ ]:
x, y = train_dataset[:]
print("\tTrain dataset length: ", len(train_dataset))
print("\tTest dataset length: ", len(test_dataset))
print("\tX, y train shapes:", x.shape, y.shape)

## Model creation

Now, the only thing left is creating the model. For this example we are using a `KAN`

In [ ]:
training_params = {
    "epochs": 2,
    "lr": 1e-5,
    'lr_gamma': 0.95,
    'lr_scheduler_step': 10,
    'batch_size': 8,
    "print_eval_rate": 1,
    "optimizer_class": torch.optim.Adam,
    "lr_kwargs":{
        "gamma": 0.95,
        "step_size": 3 * len(train_dataset) // 8 # each 3 epochs
    },
    "max_norm_grad": 0.5,
    "save_logs_path":str(CASE_DIR),
}

We can train the model on a GPU to speed up the training. pyLOM can detect if a GPU is available with ´NN.DEVICE´, that will select the fist GPU.
If many GPUs are available, the device can be define with `device = cuda:i` where `i` is the index of the GPU

In [ ]:
device = NN.DEVICE

In [ ]:
model = NN.KAN(
    input_size=x.shape[1],
    output_size=y.shape[1],
    hidden_size=31,
    n_layers=3,
    p_dropouts=0.0,
    layer_type=NN.ChebyshevLayer,
    model_name="kan_example_xfoil",
    device=device,
    degree=7
)

## Run the pipeline

In [ ]:
pipeline = NN.Pipeline(
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    model=model,
    training_params=training_params,
)
training_logs = pipeline.run()

To save the model:

In [ ]:
model.save(path=str(CASE_DIR / "models"))

## Show plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def true_vs_pred_plot(y_true, y_pred):
    """
    Auxiliary function to plot the true vs predicted values
    """
    num_plots = y_true.shape[1]
    plt.figure(figsize=(10, 5 * num_plots))
    for j in range(num_plots):
        plt.subplot(num_plots, 1, j + 1)
        plt.scatter(y_true[:, j], y_pred[:, j], s=1, c="b", alpha=0.5)
        plt.xlabel("True values")
        plt.ylabel("Predicted values")
        plt.title(f"Scatterplot for Component {j+1}")
        plt.grid(True)

    plt.tight_layout()
    plt.show()

def plot_train_test_loss(train_loss, test_loss):
    """
    Auxiliary function to plot the training and test loss
    """
    plt.figure()
    plt.plot(range(1, len(train_loss) + 1), train_loss, label="Training Loss")

    total_epochs = len(test_loss)  # test loss is calculated at the end of each epoch
    total_iters = len(train_loss)  # train loss is calculated at the end of each iteration/batch
    if total_epochs > 0 and total_iters > 0:
        iters_per_epoch = max(1, total_iters // total_epochs)
        x_test = np.arange(iters_per_epoch, iters_per_epoch * total_epochs + 1, step=iters_per_epoch)
        x_test = x_test[:total_epochs]
        plt.plot(x_test, test_loss, label="Test Loss")

    plt.xlabel("Iterations")
    plt.ylabel("Loss")
    plt.title("Training Loss vs Epoch")
    plt.yscale("log")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
plot_train_test_loss(training_logs['train_loss'], training_logs['test_loss'])

In [ ]:
preds = model.predict(train_dataset, batch_size=250)

scaled_preds = output_scaler.inverse_transform([preds])[0]
scaled_y = output_scaler.inverse_transform([train_dataset[:][1]])[0]
true_vs_pred_plot(scaled_y, scaled_preds)

## Evaluate the model with some metrics

In [ ]:
evaluator = NN.RegressionEvaluator()
evaluator(scaled_y, scaled_preds)
evaluator.print_metrics()

As you can see, this example is very similar that the one with the MLP, it just changes the model definition. This shows the power of pyLOM when training different models